# 09 · Accuracy-versus-cost trade-off

**Purpose.** Quantify the added latency and model invocations of every configuration, validate the cost model against a real online run of the gated pipeline, and characterise FP reduction per unit of added cost.

| | |
|---|---|
| **Proposal objective** | Objective 4 |
| **Inputs** | all caches |
| **Outputs** | `results/cost/*.csv`, `results/figures/cost_*.png` |
| **Runtime (T4)** | offline part: seconds; online measurement ≈2 min for 30 images |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess, shutil  # paths, sys.path, shell commands (git clone, pip), folder copy
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = (not IN_COLAB) and os.path.isdir("/kaggle/working")    # Kaggle notebook (needs Internet ON, see docs/KAGGLE_GUIDE.md)

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"
BRANCH = "kaggle-version"  # git branch to take the latest code from (change it if you push to another branch)

if IN_COLAB:
    from google.colab import drive  # Colab only: mount Google Drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, DRIVE_PROJECT], check=True)
    elif os.path.isdir(os.path.join(DRIVE_PROJECT, ".git")):   # pick up the latest pushed code (never touches your notebooks/results)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "fetch", "origin", BRANCH], check=False)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "checkout", "FETCH_HEAD", "--", "src", "configs", "requirements.txt", "tools", "app"], check=False)
    PROJECT_ROOT = DRIVE_PROJECT
elif IN_KAGGLE:
    PROJECT_ROOT = "/kaggle/working/detect-verify-decide"
    if not os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")):
        import glob
        attached = glob.glob("/kaggle/input/*/src/dvd") + glob.glob("/kaggle/input/*/*/src/dvd")   # project attached as a Kaggle Dataset
        if attached:
            shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), PROJECT_ROOT)
        elif REPO_URL:
            url = REPO_URL
            try:   # private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons -> Secrets)
                from kaggle_secrets import UserSecretsClient
                url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
            except Exception:
                pass
            r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT_ROOT], capture_output=True)
            if r.returncode:   # (the message deliberately does not echo the command, which could contain the token)
                raise RuntimeError("git clone failed: is Internet ON, REPO_URL / BRANCH right, and for a private repo the GITHUB_TOKEN secret set?")
            subprocess.run(["git", "-C", PROJECT_ROOT, "remote", "set-url", "origin", REPO_URL])   # never keep the token in .git/config
        else:
            raise RuntimeError("Set REPO_URL (and BRANCH) above, or attach the project as a Kaggle Dataset.")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab / Kaggle already have torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB or IN_KAGGLE:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt  # json files, numpy arrays, pandas tables, matplotlib plots
from dvd.config import load_config, save_selected, results_dir  # load configs/default.yaml, save tuned values, results folders
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache  # seeding, JSON helpers and the resumable JSONL cache
from dvd import paths  # cache-file path helpers

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch  # PyTorch: GPU check and memory statistics
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

In [ ]:
from dvd.data.coco import prepare_annotations, fetch_images, CocoData, load_splits  # COCO annotations, image download, splits
prepare_annotations(cfg)   # annotation json only (the 241 MB zip is cached on Drive)
coco = CocoData(cfg)
splits = load_splits(cfg)
HELDOUT, TEST = splits["heldout"], splits["test"]
fetch_images(cfg, HELDOUT + TEST)   # only these images (a few hundred files), not the whole 778 MB val2017 zip
print(f"held-out: {len(HELDOUT)} images | test: {len(TEST)} images")

In [ ]:
from dvd.experiments import SplitData, available_methods, sweep, operating_point, evaluate_row, method_cost, DET_ONLY, PROPOSED  # methods over identical candidates: sweeps, operating points, evaluation, costs
H = SplitData(cfg, coco, "heldout", HELDOUT); T = SplitData(cfg, coco, "test", TEST)
MT = available_methods(cfg, T)
fp_base = evaluate_row(MT[DET_ONLY], T, {"tau": cfg["selected"]["detector_only_tau"]})["fp"]

## 1. Cost of every (τ_low, τ_high) configuration of the pipeline

In [ ]:
rows = []
for r in sweep(PROPOSED, MT[PROPOSED], T, cfg):
    c = method_cost(MT[PROPOSED], T, r["n_verified"])
    rows.append({**r, **c, "fp_reduction": 1 - r["fp"] / fp_base})
cost_df = pd.DataFrame(rows)
cost_df.to_csv(results_dir(cfg, "cost") / "pipeline_config_costs_test.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sc = ax[0].scatter(cost_df.added_latency_s, cost_df.fp_reduction, c=cost_df.recall, cmap="viridis", s=18)
plt.colorbar(sc, ax=ax[0], label="recall"); ax[0].set(xlabel="added latency per image (s)", ylabel="FP reduction vs detector tau*", title="FP removed vs added latency")
ax[1].scatter(cost_df.verify_invocations, cost_df.f1, s=18); ax[1].set(xlabel="verifier calls per image", ylabel="F1", title="F1 vs verifier calls")
for a in ax: a.grid(alpha=.3)
plt.savefig(results_dir(cfg, "figures") / "cost_pipeline_tradeoff.png", dpi=150, bbox_inches="tight"); plt.show()

## 2. Method-level cost at each method's selected operating point

In [ ]:
main = pd.read_csv(results_dir(cfg, "tables") / "main_comparison.csv")
m = main[main.criterion == "best_f1"][["method", "f1", "fp_reduction", "det_latency_s", "added_latency_s", "total_latency_s",
                                         "det_invocations", "verify_invocations", "total_invocations"]].copy()
m["fps"] = 1 / m.total_latency_s
m["fp_reduction_per_added_s"] = m.fp_reduction / m.added_latency_s.replace(0, np.nan)
m.to_csv(results_dir(cfg, "cost") / "method_costs_test.csv", index=False); m

## 3. Validate the cost model with a real online run
The deployable pipeline verifies only the uncertain band; its measured latency should match the cost-model estimate.

In [ ]:
from dvd.agents.detector import build_detector  # Detector Agent (Grounding DINO): factory
from dvd.agents.verifier import build_verifier  # Verifier Agent (frozen Qwen2-VL): factory and crop builder
from dvd.agents.decision import RuleDecisionAgent  # Decision Agent: the accept/reject rule and the optional LLM aggregator
from dvd.pipeline import DetectVerifyDecide  # cached offline stages and the online Detect-Verify-Decide pipeline
from tqdm.auto import tqdm  # progress bars
tl, th = cfg["selected"]["tau_low"], cfg["selected"]["tau_high"]
pipe = DetectVerifyDecide(build_detector(cfg, coco.cat_names, coco.name_to_cat_id), build_verifier(cfg), RuleDecisionAgent(tl, th), cfg)
ids = TEST[: cfg["cost"]["online_latency_images"]]
pipe(coco.image(ids[0]))  # warm-up
online = []
for iid in tqdm(ids):
    r = pipe(coco.image(iid))
    online.append({"det_s": r["det_time_s"], "verify_s": r["verify_time_s"], "decide_s": r["decision_time_s"],
                   "total_s": r["total_time_s"], "verify_calls": r["verify_invocations"]})
online = pd.DataFrame(online)
est = cost_df[(cost_df.tau_low == tl) & (cost_df.tau_high == th)].iloc[0]
summary = pd.DataFrame({"measured (online, gated)": [online.det_s.mean(), online.verify_s.mean(), online.total_s.mean(), online.verify_calls.mean()],
                        "estimated (cost model)": [est.det_latency_s, est.added_latency_s, est.total_latency_s, est.verify_invocations]},
                       index=["detector s/img", "verifier s/img", "total s/img", "verifier calls/img"])
summary.to_csv(results_dir(cfg, "cost") / "online_vs_estimated.csv"); summary

In [ ]:
print(f"Online pipeline: {1 / online.total_s.mean():.2f} FPS vs detector alone {1 / online.det_s.mean():.2f} FPS "
      f"(x{online.total_s.mean() / online.det_s.mean():.2f} latency); decision agent {online.decide_s.mean() * 1e3:.3f} ms/img")
online.total_s.hist(bins=30); plt.xlabel("end-to-end seconds / image"); plt.title("Online latency distribution (bounded: single fixed pass)"); plt.show()